# Modul 06 · Notebook 00 — Ekosistem NVIDIA & GPU

Selamat datang di **modul terakhir**! Di Modul 05 kamu sudah membangun **RAG bot** yang pintar. Di modul ini kita jawab dua pertanyaan besar:

1. **Bagaimana menjalankannya dengan efisien** di hardware NVIDIA? → nb00–03
2. **Bagaimana membuatnya aman & layak dipercaya** untuk 10.000 user sungguhan? → nb04–07 (*Trustworthy AI* + *guardrails*)

### Peta perjalanan 8 notebook
| # | Notebook | Inti |
|---|----------|------|
| 00 | Ekosistem NVIDIA & GPU | GPU/CUDA, benchmark FP16 vs FP32 |
| 01 | Optimasi Inferensi | quantization 4-bit |
| 02 | NVIDIA NIM | generator cloud (tanpa GPU) |
| 03 | RAG di NIM | RAG di atas stack NVIDIA |
| 04 | Trustworthy AI & 5 Rail | 4 pilar + guardrails pertama |
| 05 | Keamanan & Privasi | jailbreak/toxicity + PII masking |
| 06 | Grounding & Topik | anti-halusinasi + topical rail |
| 07 | Capstone Deploy | service ber-guardrails penuh |

### Istilah dasar (untuk pemula)
- **GPU** (*Graphics Processing Unit*): chip dengan ribuan core kecil yang bekerja **paralel** — ideal untuk perkalian matriks di balik AI.
- **CUDA**: platform NVIDIA yang membuat GPU bisa diprogram untuk komputasi umum, bukan cuma grafis.
- **Tensor Core**: unit khusus di GPU NVIDIA yang mempercepat perkalian matriks **FP16** secara dramatis.
- **FP32 / FP16** (*floating point* 32-bit / 16-bit): presisi angka. FP16 memakai **separuh** memori.
- **Inference**: tahap **memakai** model yang sudah dilatih untuk menghasilkan output (lawan dari *training*).

> ⚙️ **Runtime:** pastikan Colab memakai **GPU T4** — menu *Runtime → Change runtime type → T4 GPU*.

In [1]:
# Instal dependensi (pinned untuk reproducibility)
!pip -q install "transformers>=4.53,<5" "accelerate>=0.30"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.5 MB/s eta 0:00:00


In [2]:
# Cek GPU — Cara 1: CLI bawaan NVIDIA
!nvidia-smi

Mon Jun 15 07:18:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Cek GPU — Cara 2: lewat PyTorch
import torch
print("CUDA tersedia :", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("Nama GPU      :", torch.cuda.get_device_name(0))
    print("Compute cap.  :", torch.cuda.get_device_capability(0))
    print("Memori total  :", round(p.total_memory / 1e9, 1), "GB")
    print("Memori dipakai:", round(torch.cuda.memory_allocated(0) / 1e9, 2), "GB")
else:
    print("GPU tidak aktif - aktifkan T4 via Runtime -> Change runtime type.")

CUDA tersedia : True
Nama GPU      : Tesla T4
Compute cap.  : (7, 5)
Memori total  : 15.6 GB
Memori dipakai: 0.0 GB


## FP16 vs FP32 — apa untungnya separuh presisi?

Kita uji model yang **sama** (Qwen2.5-1.5B) pada dua presisi dan bandingkan **tiga hal**: memori, kualitas output, dan kecepatan. Hasilnya akan menunjukkan kapan FP16 menang — dan kapan tidak.

In [4]:
# Muat model dalam satu presisi: ukur memori bobot + hasilkan satu jawaban (cek kualitas)
import torch, time, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)

def load_and_answer(dtype, prompt="Jelaskan apa itu GPU dalam satu kalimat sederhana."):
    torch.cuda.empty_cache(); gc.collect()
    model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=dtype, device_map="auto")
    mem_gb = torch.cuda.memory_allocated() / 1e9                       # memori bobot model
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                  add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=64,
                         do_sample=False, pad_token_id=tok.eos_token_id)
    txt = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    del model; gc.collect(); torch.cuda.empty_cache()
    return round(mem_gb, 2), txt

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
mem32, txt32 = load_and_answer(torch.float32)
mem16, txt16 = load_and_answer(torch.float16)

print(f"Memori model FP32 : {mem32:.2f} GB")
print(f"Memori model FP16 : {mem16:.2f} GB   -> {mem32/mem16:.1f}x lebih hemat\n")
print("Output FP32:", txt32)
print("Output FP16:", txt16)
print("\nOutput identik?", txt32 == txt16)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Memori model FP32 : 6.18 GB
Memori model FP16 : 3.10 GB   -> 2.0x lebih hemat

Output FP32: GPU adalah singkatan dari Graphics Processing Unit, yang merupakan komponen hardware utama dalam sistem komputer yang dirancang untuk mengatasi tugas grafis dan visualisasi dengan cepat dan efektif.
Output FP16: GPU adalah singkatan dari Graphics Processing Unit, yang merupakan komponen hardware utama dalam sistem komputer yang dirancang untuk mengatasi tugas grafis dan visualisasi dengan cepat dan efektif.

Output identik? True


## Lalu... kecepatan?

Memori FP16 jelas separuh, dan output nyaris sama. Tapi **di mana FP16 lebih cepat?** Jawabannya: pada **komputasi berat** seperti perkalian matriks besar, yang dipercepat oleh **Tensor Core** GPU NVIDIA. Mari ukur.

In [6]:
# Kecepatan: perkalian matriks 4096x4096 (FP32 vs FP16) -> Tensor Core menyala untuk FP16
def matmul_bench(dtype, n=4096, iters=50):
    a = torch.randn(n, n, device="cuda", dtype=dtype)
    b = torch.randn(n, n, device="cuda", dtype=dtype)
    for _ in range(5):                       # warmup
        _ = a @ b
    torch.cuda.synchronize()                 # GPU async -> wajib sinkron sebelum mengukur
    t0 = time.time()
    for _ in range(iters):
        _ = a @ b
    torch.cuda.synchronize()
    return (time.time() - t0) / iters * 1000  # milidetik per matmul

ms32 = matmul_bench(torch.float32)
ms16 = matmul_bench(torch.float16)
print(f"Matmul 4096x4096 FP32 : {ms32:6.2f} ms")
print(f"Matmul 4096x4096 FP16 : {ms16:6.2f} ms   -> {ms32/ms16:.1f}x lebih cepat (Tensor Core)")

Matmul 4096x4096 FP32 :  34.81 ms
Matmul 4096x4096 FP16 :   6.23 ms   -> 5.6x lebih cepat (Tensor Core)


## Yang kita pelajari

- **Memori:** FP16 = **separuh** FP32 (selalu) → bisa memuat model yang lebih besar di GPU yang sama.
- **Kualitas:** output FP16 ≈ FP32 (nyaris identik) — presisi yang hilang hampir tak terasa.
- **Kecepatan:** FP16 **jauh lebih cepat untuk komputasi berat** (matmul lewat *Tensor Core*).

> 💡 **Catatan jujur:** untuk **generate teks token-demi-token** dengan 1 input, kecepatan FP16 ≈ FP32 (kadang malah sedikit lebih lambat) karena bottleneck-nya **bandwidth memori**, bukan komputasi — Tensor Core tidak banyak terpakai. Keunggulan kecepatan FP16 paling terasa pada **komputasi besar / batch besar**, persis seperti benchmark matmul di atas. Jadi: keuntungan FP16 yang **paling pasti** adalah **memori** (dan memuat model lebih besar).

Di **nb01** kita melangkah lebih jauh: **quantization 4-bit**, yang memangkas memori jauh lebih banyak lagi.

## 🧪 Latihan

1. Ganti `NAME` menjadi `"Qwen/Qwen2.5-0.5B-Instruct"` lalu jalankan ulang — bagaimana memorinya berubah?
2. Pada `matmul_bench`, ubah `n` dari 4096 menjadi 1024. Apakah keunggulan FP16 mengecil? (Petunjuk: matriks kecil kurang "memberi makan" Tensor Core.)
3. Tambah presisi `torch.bfloat16` pada `matmul_bench`. Di T4 (Turing), apakah secepat FP16? (Catatan: `bfloat16` baru dipercepat penuh di GPU Ampere ke atas.)